# DKD Risk Prediction in Type 2 Diabetes
## 03C. Clinical Benchmark and Algorithmic Sensitivity

This notebook fits additional one-year landmark comparator models using the
finalized corrected cohort and the same frozen train/calibration/test partitions
used for the primary XGBoost models.

Two analyses are performed:

1. **Parsimonious clinical benchmark:** logistic regression using age at T2D
   documentation, eGFR, UACR, and HbA1c.
2. **Algorithmic sensitivity:** logistic regression and random forest using the
   full one-year M5 feature set.

Hyperparameter optimization is restricted to the training partition. Isotonic
calibration is fitted on the calibration partition. No test-set information is used
for model fitting, calibration, or threshold selection.


# 1. Setup


In [ ]:
import json
import pickle
import warnings

import gcsfs
import numpy as np
import optuna
import pandas as pd

from sklearn.base import clone
from sklearn.calibration import CalibratedClassifierCV
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

migration_folder = "rw-migration-aou-rw-639fade3"
bucket = migration_folder

COHORT_DIR = f"{bucket}/aou_dkd/dkd_cohort_20260821"
MODEL_DIR = f"{bucket}/aou_dkd/dkd_models_20260822"
RESULT_DIR = f"{bucket}/aou_dkd/dkd_model_results_20260822"

fs = gcsfs.GCSFileSystem()

ID_COL = "person_id"
Y_COL = "dkd_binary"
FINAL_DATE = "20260822"

RANDOM_SEED = 42
CV_FOLDS = 5
N_TRIALS_LR = 20
N_TRIALS_RF = 20

# 96-CPU machine: parallelize folds while limiting threads per RF.
CV_N_JOBS = 5
RF_CV_N_JOBS = 16
RF_FINAL_N_JOBS = 64


## I/O helpers


In [ ]:
def read_gcs_pickle(path):
    with fs.open(path, "rb") as f:
        return pd.read_pickle(f)

def read_gcs_csv(path):
    with fs.open(path, "r") as f:
        return pd.read_csv(f)

def save_gcs_pickle(obj, path):
    with fs.open(path, "wb") as f:
        pickle.dump(obj, f)
    print(f"Saved pickle: {path}")

def save_gcs_csv(df, path):
    with fs.open(path, "w") as f:
        df.to_csv(f, index=False)
    print(f"Saved CSV: {path}")

def save_gcs_json(obj, path):
    with fs.open(path, "w") as f:
        json.dump(obj, f, indent=2)
    print(f"Saved JSON: {path}")


# 2. Load the Final One-Year Cohort, Split Manifest, and M5 Feature Manifest


In [ ]:
DATA_PATH = (
    f"{COHORT_DIR}/"
    f"DKD_Risk_FinalModel_Data_Landmark_1y_{FINAL_DATE}.pkl"
)

SPLIT_PATH = (
    f"{RESULT_DIR}/"
    f"final_model_split_manifest_{FINAL_DATE}.csv"
)

FEATURE_PATH = (
    f"{RESULT_DIR}/"
    f"final_model_feature_manifest_{FINAL_DATE}.csv"
)

for path in [DATA_PATH, SPLIT_PATH, FEATURE_PATH]:
    if not fs.exists(path):
        raise FileNotFoundError(path)

df_1y = read_gcs_pickle(DATA_PATH)
split_manifest = read_gcs_csv(SPLIT_PATH)
feature_manifest = read_gcs_csv(FEATURE_PATH)

df_1y[ID_COL] = pd.to_numeric(
    df_1y[ID_COL],
    errors="raise",
).astype("int64")

df_1y[Y_COL] = pd.to_numeric(
    df_1y[Y_COL],
    errors="raise",
).astype(int)

split_1y = split_manifest.loc[
    split_manifest["landmark"].eq("1y")
].copy()

split_1y[ID_COL] = pd.to_numeric(
    split_1y[ID_COL],
    errors="raise",
).astype("int64")

if df_1y[ID_COL].duplicated().any():
    raise ValueError("Duplicate person_id in one-year data.")

if split_1y[ID_COL].duplicated().any():
    raise ValueError("Duplicate person_id in one-year split manifest.")

if set(df_1y[ID_COL]) != set(split_1y[ID_COL]):
    raise ValueError(
        "One-year data and split manifest contain different participants."
    )

m5_features = (
    feature_manifest.loc[
        feature_manifest["landmark"].eq("1y")
        & feature_manifest["model_id"].eq("M5")
    ]
    .sort_values("feature_order")["feature"]
    .tolist()
)

if not m5_features:
    raise ValueError("No one-year M5 feature manifest found.")

missing_m5 = [
    f for f in m5_features
    if f not in df_1y.columns
]

if missing_m5:
    raise ValueError(
        f"One-year M5 data missing manifest predictors: {missing_m5}"
    )

print(
    f"1y corrected cohort: N={len(df_1y):,} | "
    f"cases={int(df_1y[Y_COL].sum()):,}"
)

print(f"M5 predictors: {len(m5_features):,}")


# 3. Resolve the Four-Variable Clinical Benchmark


In [ ]:
CLINICAL_CANDIDATES = {
    "Age": [
        "age_at_t2d",
        "age_at_t2d_diagnosis",
    ],
    "eGFR": [
        "eGFR_last",
        "egfr_last",
    ],
    "UACR": [
        "UACR_last",
        "uacr_last",
    ],
    "HbA1c": [
        "HbA1c_last",
        "hba1c_last",
    ],
}

def resolve_one(label, candidates, columns):
    present = [
        c for c in candidates
        if c in columns
    ]

    if len(present) != 1:
        raise ValueError(
            f"{label}: expected exactly one available feature; "
            f"found {present}. Candidates={candidates}"
        )

    return present[0]

clinical_feature_map = {
    label: resolve_one(
        label,
        candidates,
        df_1y.columns,
    )
    for label, candidates in CLINICAL_CANDIDATES.items()
}

clinical_features = list(
    clinical_feature_map.values()
)

print("Clinical benchmark features:")
for label, feature in clinical_feature_map.items():
    print(f"  {label:<6} -> {feature}")

if len(clinical_features) != 4:
    raise ValueError("Clinical benchmark must contain exactly four predictors.")


# 4. Reconstruct the Frozen Partitions


In [ ]:
split_lookup = {
    int(pid): split
    for pid, split in zip(
        split_1y[ID_COL],
        split_1y["split"],
    )
}

def get_partition(df, split_name):
    ids = {
        pid
        for pid, split in split_lookup.items()
        if split == split_name
    }

    out = df.loc[
        df[ID_COL].astype(int).isin(ids)
    ].copy()

    return (
        out.sort_values(ID_COL)
        .reset_index(drop=True)
    )

train_df = get_partition(df_1y, "train")
cal_df = get_partition(df_1y, "cal")
test_df = get_partition(df_1y, "test")

for name, part in [
    ("train", train_df),
    ("cal", cal_df),
    ("test", test_df),
]:
    print(
        f"{name:>5}: N={len(part):,} | "
        f"cases={int(part[Y_COL].sum()):,} | "
        f"controls={int((part[Y_COL] == 0).sum()):,}"
    )

if (
    len(train_df)
    + len(cal_df)
    + len(test_df)
    != len(df_1y)
):
    raise ValueError("Partition sizes do not sum to the one-year cohort.")


# 5. Training Helpers


In [ ]:
def make_lr_pipeline(
    C=1.0,
    *,
    add_indicator=True,
):
    return Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="median",
                    add_indicator=add_indicator,
                ),
            ),
            (
                "scaler",
                StandardScaler(),
            ),
            (
                "model",
                LogisticRegression(
                    C=float(C),
                    penalty="l2",
                    solver="lbfgs",
                    max_iter=5000,
                    random_state=RANDOM_SEED,
                ),
            ),
        ]
    )

def make_rf_pipeline(params, *, n_jobs):
    return Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="median",
                    add_indicator=True,
                ),
            ),
            (
                "model",
                RandomForestClassifier(
                    n_estimators=int(params["n_estimators"]),
                    max_depth=int(params["max_depth"]),
                    min_samples_leaf=int(params["min_samples_leaf"]),
                    max_features=float(params["max_features"]),
                    n_jobs=n_jobs,
                    random_state=RANDOM_SEED,
                ),
            ),
        ]
    )

def fit_isotonic_and_save_predictions(
    model,
    features,
    *,
    model_key,
    model_filename,
    calibrator_filename,
    prediction_filename,
):
    X_train = train_df[features]
    y_train = train_df[Y_COL].to_numpy()

    X_cal = cal_df[features]
    y_cal = cal_df[Y_COL].to_numpy()

    X_test = test_df[features]
    y_test = test_df[Y_COL].to_numpy()

    model.fit(
        X_train,
        y_train,
    )

    calibrator = CalibratedClassifierCV(
        model,
        method="isotonic",
        cv="prefit",
    )

    calibrator.fit(
        X_cal,
        y_cal,
    )

    model_path = f"{MODEL_DIR}/{model_filename}"
    calibrator_path = f"{MODEL_DIR}/{calibrator_filename}"
    prediction_path = f"{RESULT_DIR}/{prediction_filename}"

    save_gcs_pickle(
        model,
        model_path,
    )

    save_gcs_pickle(
        calibrator,
        calibrator_path,
    )

    rows = []

    for split_name, part, X, y in [
        ("cal", cal_df, X_cal, y_cal),
        ("test", test_df, X_test, y_test),
    ]:
        rows.append(
            pd.DataFrame(
                {
                    ID_COL: part[ID_COL].astype(int).to_numpy(),
                    "split": split_name,
                    "y_true": y,
                    "p_raw": model.predict_proba(X)[:, 1],
                    "p_calibrated": calibrator.predict_proba(X)[:, 1],
                }
            )
        )

    pred = pd.concat(
        rows,
        ignore_index=True,
    )

    save_gcs_csv(
        pred,
        prediction_path,
    )

    return {
        "model_key": model_key,
        "model_path": model_path,
        "calibrator_path": calibrator_path,
        "prediction_path": prediction_path,
        "n_features": len(features),
    }


# 6. Clinical Benchmark: Four-Variable Logistic Regression


In [ ]:
X_train_clin = train_df[clinical_features]
y_train = train_df[Y_COL].to_numpy()

cv = StratifiedKFold(
    n_splits=CV_FOLDS,
    shuffle=True,
    random_state=RANDOM_SEED,
)

def clinical_lr_objective(trial):
    C = trial.suggest_float(
        "C",
        1e-4,
        1e4,
        log=True,
    )

    model = make_lr_pipeline(
        C=C,
        add_indicator=False,
    )

    scores = cross_val_score(
        model,
        X_train_clin,
        y_train,
        cv=cv,
        scoring="roc_auc",
        n_jobs=CV_N_JOBS,
    )

    return float(scores.mean())

clinical_study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(
        seed=RANDOM_SEED
    ),
)

clinical_study.optimize(
    clinical_lr_objective,
    n_trials=N_TRIALS_LR,
)

clinical_best = dict(
    clinical_study.best_params
)

clinical_model = make_lr_pipeline(
    C=clinical_best["C"],
    add_indicator=False,
)

clinical_result = fit_isotonic_and_save_predictions(
    clinical_model,
    clinical_features,
    model_key="clinical_lr",
    model_filename=(
        f"dkd_1y_clinical_lr_raw_{FINAL_DATE}.pkl"
    ),
    calibrator_filename=(
        f"dkd_1y_clinical_lr_isotonic_calibrator_{FINAL_DATE}.pkl"
    ),
    prediction_filename=(
        f"dkd_1y_clinical_lr_predictions_{FINAL_DATE}.csv"
    ),
)

print(
    f"Clinical LR best CV AUROC: "
    f"{clinical_study.best_value:.5f}"
)

print("Clinical LR best params:", clinical_best)


# 7. Full-M5 Logistic Regression


In [ ]:
X_train_m5 = train_df[m5_features]

def m5_lr_objective(trial):
    C = trial.suggest_float(
        "C",
        1e-4,
        1e4,
        log=True,
    )

    model = make_lr_pipeline(
        C=C,
        add_indicator=True,
    )

    scores = cross_val_score(
        model,
        X_train_m5,
        y_train,
        cv=cv,
        scoring="roc_auc",
        n_jobs=CV_N_JOBS,
    )

    return float(scores.mean())

lr_study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(
        seed=RANDOM_SEED
    ),
)

lr_study.optimize(
    m5_lr_objective,
    n_trials=N_TRIALS_LR,
)

lr_best = dict(
    lr_study.best_params
)

lr_model = make_lr_pipeline(
    C=lr_best["C"],
    add_indicator=True,
)

lr_result = fit_isotonic_and_save_predictions(
    lr_model,
    m5_features,
    model_key="m5_lr",
    model_filename=(
        f"dkd_1y_M5_lr_raw_{FINAL_DATE}.pkl"
    ),
    calibrator_filename=(
        f"dkd_1y_M5_lr_isotonic_calibrator_{FINAL_DATE}.pkl"
    ),
    prediction_filename=(
        f"dkd_1y_M5_lr_predictions_{FINAL_DATE}.csv"
    ),
)

print(
    f"M5 LR best CV AUROC: "
    f"{lr_study.best_value:.5f}"
)

print("M5 LR best params:", lr_best)


# 8. Full-M5 Random Forest


In [ ]:
def rf_objective(trial):
    params = {
        "n_estimators": trial.suggest_int(
            "n_estimators",
            300,
            900,
            step=100,
        ),
        "max_depth": trial.suggest_int(
            "max_depth",
            6,
            30,
        ),
        "min_samples_leaf": trial.suggest_int(
            "min_samples_leaf",
            1,
            20,
        ),
        "max_features": trial.suggest_float(
            "max_features",
            0.20,
            1.00,
        ),
    }

    model = make_rf_pipeline(
        params,
        n_jobs=RF_CV_N_JOBS,
    )

    scores = cross_val_score(
        model,
        X_train_m5,
        y_train,
        cv=cv,
        scoring="roc_auc",
        n_jobs=CV_N_JOBS,
    )

    return float(scores.mean())

rf_study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(
        seed=RANDOM_SEED
    ),
)

rf_study.optimize(
    rf_objective,
    n_trials=N_TRIALS_RF,
)

rf_best = dict(
    rf_study.best_params
)

rf_model = make_rf_pipeline(
    rf_best,
    n_jobs=RF_FINAL_N_JOBS,
)

rf_result = fit_isotonic_and_save_predictions(
    rf_model,
    m5_features,
    model_key="m5_rf",
    model_filename=(
        f"dkd_1y_M5_rf_raw_{FINAL_DATE}.pkl"
    ),
    calibrator_filename=(
        f"dkd_1y_M5_rf_isotonic_calibrator_{FINAL_DATE}.pkl"
    ),
    prediction_filename=(
        f"dkd_1y_M5_rf_predictions_{FINAL_DATE}.csv"
    ),
)

print(
    f"M5 RF best CV AUROC: "
    f"{rf_study.best_value:.5f}"
)

print("M5 RF best params:", rf_best)


# 9. Save Comparator Training Summary


In [ ]:
summary_rows = [
    {
        "model_key": "clinical_lr",
        "analysis": "clinical_benchmark",
        "n_features": len(clinical_features),
        "best_cv_auroc": float(clinical_study.best_value),
        "best_params": json.dumps(clinical_best),
    },
    {
        "model_key": "m5_lr",
        "analysis": "algorithmic_sensitivity",
        "n_features": len(m5_features),
        "best_cv_auroc": float(lr_study.best_value),
        "best_params": json.dumps(lr_best),
    },
    {
        "model_key": "m5_rf",
        "analysis": "algorithmic_sensitivity",
        "n_features": len(m5_features),
        "best_cv_auroc": float(rf_study.best_value),
        "best_params": json.dumps(rf_best),
    },
]

summary_df = pd.DataFrame(
    summary_rows
)

save_gcs_csv(
    summary_df,
    f"{RESULT_DIR}/"
    f"comparator_training_summary_{FINAL_DATE}.csv",
)

save_gcs_json(
    {
        "date": FINAL_DATE,
        "random_seed": RANDOM_SEED,
        "cv_folds": CV_FOLDS,
        "lr_trials": N_TRIALS_LR,
        "rf_trials": N_TRIALS_RF,
        "clinical_features": clinical_feature_map,
        "m5_n_features": len(m5_features),
        "clinical_lr": clinical_result,
        "m5_lr": lr_result,
        "m5_rf": rf_result,
    },
    f"{RESULT_DIR}/"
    f"comparator_training_summary_{FINAL_DATE}.json",
)

display(summary_df)

print("=" * 90)
print("NOTEBOOK 03C COMPLETE")
print("=" * 90)
